In [0]:
from tabulate import tabulate

In [0]:
customers_table = "workspace.default.customers"
orders_table = "workspace.default.orders"
order_items_table = "workspace.default.order_items"
products_table = "workspace.default.products"

In [0]:
def revenue_report():

    query = f"""
    SELECT
        ROUND(
            SUM(
                quantity * unit_price
                * (1 - discount_percent / 100)
            ),
            2
        ) AS total_revenue
    FROM {order_items_table}
    """

    result = spark.sql(query)

    rows = result.collect()

    if not rows:
        print("No revenue data found.")
        return

    data = [row.asDict() for row in rows]

    print(tabulate(
        data,
        headers="keys",
        tablefmt="grid"
    ))

In [0]:
def top_customers_report():

    query = f"""
    SELECT
        c.customer_id,
        c.customer_name,

        ROUND(
            SUM(
                oi.quantity * oi.unit_price
                * (1 - oi.discount_percent / 100)
            ),
            2
        ) AS total_spend

    FROM {customers_table} c

    JOIN {orders_table} o
        ON c.customer_id = o.customer_id

    JOIN {order_items_table} oi
        ON o.order_id = oi.order_id

    GROUP BY
        c.customer_id,
        c.customer_name

    ORDER BY total_spend DESC

    LIMIT 10
    """

    result = spark.sql(query)

    rows = result.collect()

    if not rows:
        print("No customer data found.")
        return

    data = [row.asDict() for row in rows]

    print(tabulate(
        data,
        headers="keys",
        tablefmt="grid"
    ))

In [0]:
def monthly_revenue_report():

    query = f"""
    SELECT
        DATE_TRUNC('month', o.order_date) AS month,

        ROUND(
            SUM(
                oi.quantity * oi.unit_price
                * (1 - oi.discount_percent / 100)
            ),
            2
        ) AS revenue

    FROM {orders_table} o

    JOIN {order_items_table} oi
        ON o.order_id = oi.order_id

    GROUP BY
        DATE_TRUNC('month', o.order_date)

    ORDER BY month
    """

    result = spark.sql(query)

    rows = result.collect()

    if not rows:
        print("No monthly revenue data found.")
        return

    data = [row.asDict() for row in rows]

    print(tabulate(
        data,
        headers="keys",
        tablefmt="grid"
    ))

In [0]:
def top_products_report():

    query = f"""
    SELECT
        p.product_id,
        p.product_name,

        SUM(oi.quantity) AS quantity_sold

    FROM {products_table} p

    JOIN {order_items_table} oi
        ON p.product_id = oi.product_id

    GROUP BY
        p.product_id,
        p.product_name

    ORDER BY quantity_sold DESC

    LIMIT 10
    """

    result = spark.sql(query)

    rows = result.collect()

    if not rows:
        print("No product data found.")
        return

    data = [row.asDict() for row in rows]

    print(tabulate(
        data,
        headers="keys",
        tablefmt="grid"
    ))

In [0]:
def customer_segment_report():

    query = f"""
    SELECT
        c.customer_id,
        c.customer_name,

        COUNT(DISTINCT o.order_id) AS total_orders,

        ROUND(
            COALESCE(
                SUM(
                    oi.quantity * oi.unit_price
                    * (1 - oi.discount_percent / 100)
                ),
                0
            ),
            2
        ) AS total_spend,

        CASE
            WHEN COUNT(DISTINCT o.order_id) = 0
                THEN 'No Purchase'

            WHEN COUNT(DISTINCT o.order_id) = 1
                THEN 'One-time'

            WHEN COUNT(DISTINCT o.order_id) <= 4
                THEN 'Occasional'

            ELSE 'Loyal'
        END AS customer_segment

    FROM {customers_table} c

    LEFT JOIN {orders_table} o
        ON c.customer_id = o.customer_id

    LEFT JOIN {order_items_table} oi
        ON o.order_id = oi.order_id

    GROUP BY
        c.customer_id,
        c.customer_name

    ORDER BY total_spend DESC
    """

    result = spark.sql(query)

    rows = result.collect()

    if not rows:
        print("No customer segment data found.")
        return

    data = [row.asDict() for row in rows]

    print(tabulate(
        data,
        headers="keys",
        tablefmt="grid"
    ))

In [0]:
print("E-Commerce Analytics Reports")
print("----------------------------")
print("1. Revenue")
print("2. Top Customers")
print("3. Monthly Revenue")
print("4. Top Products")
print("5. Customer Segments")

choice = input("Enter your choice: ")

E-Commerce Analytics Reports
----------------------------
1. Revenue
2. Top Customers
3. Monthly Revenue
4. Top Products
5. Customer Segments


Enter your choice:  1

In [0]:
if choice == "1":
    revenue_report()

elif choice == "2":
    top_customers_report()

elif choice == "3":
    monthly_revenue_report()

elif choice == "4":
    top_products_report()

elif choice == "5":
    customer_segment_report()

else:
    print("Invalid choice. Please select 1 to 5.")

+-----------------+
|   total_revenue |
+=================+
|     6.40639e+07 |
+-----------------+
